In [1]:
import pandas as pd
import numpy as np
import torch
import plotly.graph_objects as go
import matplotlib.pyplot as plt

from src.utils import generate_mask_tensor
from src.embedding import embed
from src.gp_ccm import GP_ccm_sig, run_sigGPCCM_experiment
from src.sp_ccm import run_SP_CCM, SP_CCM_iaaft, run_ccm_experiment
from src.iaaft import surrogates

from scipy.stats import ranksums
torch.set_printoptions(sci_mode = False)

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)
print()

Using device: cuda



# Generate noisy data

In [10]:
# Set length of timeseries
N_length = torch.tensor([400])

# Initialise values at t = 0
f = torch.tensor([0.2])
g = torch.tensor([0.4])

# In original domain
f_noise = torch.tensor([0.2])
g_noise = torch.tensor([0.2])


# Autoregressive function
for t in range(N_length - 1):
    
    # from ECCM
    f_next = (f[t] * (3.8 - (3.8 * f[t])))
    g_next = (g[t] * (3.1 - (3.1 * g[t]) - (0.8 * f[t])))

    f = torch.concat((f, f_next.unsqueeze(0)))
    g = torch.concat((g, g_next.unsqueeze(0)))

f = f + torch.randn(size = (N_length, )).mul(f_noise)
g = g + torch.randn(size = (N_length, )).mul(g_noise)
            
# Normalising step
f_norm = f.sub(f.mean(dim = -1).unsqueeze(-1)).div(f.std(dim = -1).unsqueeze(-1))
g_norm = g.sub(g.mean(dim = -1).unsqueeze(-1)).div(g.std(dim = -1).unsqueeze(-1))

# Other 

In [62]:
# Set length of timeseries
N_length = torch.tensor([400])

# Initialise values at t = 0
f = torch.tensor([0.2])
g = torch.tensor([0.4])

# In original domain
f_noise = torch.tensor([0.0])
g_noise = torch.tensor([0.0])


# Autoregressive function
for t in range(N_length - 1):
    
    # from ECCM
    f_next = torch.sin(torch.tensor([t - 0.5 * t])) + 0.005 * t
    g_next = (g[t] * (3.1 - (3.1 * g[t]) - (0.2 * f[t])))

    f = torch.concat((f, f_next))
    g = torch.concat((g, g_next.unsqueeze(0)))

f = f + torch.randn(size = (N_length, )).mul(f_noise)
g = g + torch.randn(size = (N_length, )).mul(g_noise)
            
# Normalising step
f_norm = f.sub(f.mean(dim = -1).unsqueeze(-1)).div(f.std(dim = -1).unsqueeze(-1))
g_norm = g.sub(g.mean(dim = -1).unsqueeze(-1)).div(g.std(dim = -1).unsqueeze(-1))

In [63]:
MAX = 100

fig = go.Figure()

fig.add_trace(go.Scatter(x = torch.arange(0, f.shape[0])[0:MAX], y = f[0:MAX],
                    mode = 'lines',
                    name = 'F',
                    line_color = "red"))

fig.add_trace(go.Scatter(x = torch.arange(0, g.shape[0])[0:MAX], y = g[0:MAX],
                    mode = 'lines',
                    name = 'G',
                    line_color = "purple"))

fig.update_layout(title = 'Noisy time series',
                   xaxis_title = 't',
                   yaxis_title = 'values')

fig.update_layout(template = "plotly_white")
fig.update_layout(font_family = "Lato")
fig.update_layout(xaxis_range=[-2, MAX])

fig.update_layout(autosize = False, width = 1000, height = 400)

fig.show()

In [64]:
MAX = 400

fig = go.Figure()

fig.add_trace(go.Scatter(x = torch.arange(0, f_norm.shape[0])[0:MAX], y = f_norm[0:MAX],
                    mode = 'lines',
                    name = 'F',
                    line_color = "red"))

fig.add_trace(go.Scatter(x = torch.arange(0, g_norm.shape[0])[0:MAX], y = g_norm[0:MAX],
                    mode = 'lines',
                    name = 'G',
                    line_color = "purple"))

fig.update_layout(title = 'Noisy time series',
                   xaxis_title = 't',
                   yaxis_title = 'values')

fig.update_layout(template = "plotly_white")
fig.update_layout(font_family = "Lato")
fig.update_layout(xaxis_range=[-2, MAX])

fig.update_layout(autosize = False, width = 1000, height = 400)

fig.show()

In [65]:
# GLOBALS
k = 3
N_TRAIN = torch.tensor([100]).to(device)

##############
### GP-CCM ###
##############

sig_filter = torch.ones(size = (k, )).to(device)
sig_shift = torch.tensor(sig_filter.shape[0] - 1).to(device) # k -1 

NOISE_SCALE = torch.tensor([0.05], device = device) # for diagonal
RBF_SCALE = torch.tensor([0.3], device = device)

############
### ECCM ###
############

ccm_filter = torch.ones(size = (k, )).to(device) # same as sig filter
ccm_shift = torch.tensor(ccm_filter.shape[0] - 1).to(device)

# F -> G (true)

In [66]:
### sig-GP_CCM ###
FG_gpccm_rho_mean, FG_gpccm_rho_sd, FG_gpccm_rho_ind_p95 =  run_sigGPCCM_experiment(
    causal_x = f_norm.to(device),
    causal_y = g_norm.to(device),
    sig_filter = sig_filter.to(device),
    sig_shift = sig_shift.to(device),
    rbf_scale = RBF_SCALE, 
    noise_scale = NOISE_SCALE, 
    n_train = N_TRAIN.to(device),
    device = device)

Rho mean 0.32
Rho std 0.12
Rho indep. p95 0.393


In [67]:
### CCM ###
FG_ccm_rho_mean, FG_ccm_rho_sd, FG_ccm_rho_ind_p95, FG_ccm_noise =  run_ccm_experiment(
    causal_x = f_norm.to(device),
    causal_y = g_norm.to(device),
    ccm_filter = ccm_filter.to(device),
    ccm_shift = ccm_shift.to(device),
    n_train = N_TRAIN.to(device),
    device = device)

Rho mean 0.956
Rho std 0.02
Rho indep. p95 0.669
Added noise 0.0


# G -> F (false)

In [57]:
### sig-GP_CCM ###
GF_gpccm_rho_mean, GF_gpccm_rho_sd, GF_gpccm_rho_ind_p95 =  run_sigGPCCM_experiment(
    causal_x = g_norm.to(device),
    causal_y = f_norm.to(device),
    sig_filter = sig_filter.to(device),
    sig_shift = sig_shift.to(device),
    rbf_scale = RBF_SCALE, 
    noise_scale = NOISE_SCALE, 
    n_train = N_TRAIN.to(device),
    device = device)

Rho mean 0.483
Rho std 0.203
Rho indep. p95 0.456


In [58]:
### CCM ###
GF_ccm_rho_mean, GF_ccm_rho_sd, GF_ccm_rho_ind_p95, GF_ccm_noise =  run_ccm_experiment(
    causal_x = g_norm.to(device),
    causal_y = f_norm.to(device),
    ccm_filter = ccm_filter.to(device),
    ccm_shift = ccm_shift.to(device),
    n_train = N_TRAIN.to(device),
    device = device)

Rho mean 0.923
Rho std 0.024
Rho indep. p95 0.783
Added noise 0.0
